# English STELLA Transcriptions Dataset

The STELLA dataset...


# Data preparation

Format Dataset into the wanted architecture. This procedure extracts audiobook transcriptions from the original dataset and sorts them into the same splits as the audio files.

```
txt
├── LANG
│   ├── HOUR_SPLIT
│   │   ├── SECTION_SPLIT
│   │   │   ├── books.txt
│   │   │   ├── meta.json
│   │   │   └── transcription.txt
│   │   ├── ...
│   ├── ...
│   ...

```
- txt : folder containing transcriptions
- LANG: corresponds to the given language
- HOUR_SPLIR: corresponds to the size of the section splits in number of hours of speech,
              formatted as (50h, 100h, ..., 3200h)
- SECTION_SPLIT: separation of content into sections with equal amount of speech content.
- books.txt: the list of books used for this split
- meta.json: metadata generated during clean-up used to measure effectiveness of cleaning.
- transcript.txt: the agregated transcripts of the audiobooks in the list.

In [ ]:
import platform

from lexical_benchmark import settings
from lexical_benchmark.datasets import stella
from lexical_benchmark.utils import timed_status

assert (
    platform.node() in settings.PATH.KNOWN_HOSTS
), "Not in a known device, must provide custom PATH locations"

prep = stella.STELAPrepTranscripts(lang="EN")
with timed_status(status="Preping stela transcriptions...", complete_status="Succesfuly build STELA Transcript dataset !"):
    prep.build_transcript()

## Data Cleaning

Clean-up text to keep only clean words that can be piped through the dictionairy.

RULES (Order Matters):
1) Illustration tag removal
2) URL removal
3) TextNormalisation : correct accents & remove non-printable characters
4) Trancribe numbers
5) Remove roman numerals
6) Fix symbols ($,€, etc..)
7) AZFilter

    * replace '-' with a space to extract hyphenated words (fifty-five -> fifty five)

    * Keeps apostrophe char(*'*) to protect shorthands (ex: ain't)
  
    * purges everything not between [A-Z].

    * lowecases everything
8) Fix words by removing prefix and trailing quote char (')

In [ ]:
import platform

from lexical_benchmark import settings
from lexical_benchmark.datasets import stella
from lexical_benchmark.utils import DatasetCleaner, timed_status

assert (
    platform.node() in settings.PATH.KNOWN_HOSTS
), "Not in a known device, must provide custom PATH locations"

dataset = stella.STELATranscriptDataset()
with timed_status(status="Cleaning STELA Transcripts", complete_status="Succesfuly cleanned up STELA Transcript dataset !"):
    DatasetCleaner.cleanup_files(
        filemap=dataset.raw2clean_filesmap("EN"),
        ruleset=dataset.clean_up_rules("EN"),
        save_logs=True
    )

# Word Filtering

Using a pre-selected lexicon we filter the corpus to separated known from unknown words

In [ ]:
import platform

from lexical_benchmark import settings
from lexical_benchmark.datasets import stella
from lexical_benchmark.datasets import utils as dataset_utils
from lexical_benchmark.utils import timed_status

assert (
    platform.node() in settings.PATH.KNOWN_HOSTS
), "Not in a known device, must provide custom PATH locations"

dataset = stella.STELATranscriptDataset()

with timed_status(status="Word Filtering", complete_status="Succesfuly completed word filtering !"):
    dataset_utils.DatasetCleaner.word_validate_files(
        filemap=dataset.word_validation_filesmap("EN"),
        cleaner=dataset_utils.DictionairyCleaner(lang="EN"),
    )

## Compute word frequency maps

To allow statistics on word cleaning we generate word_frequency table for all steps of the cleaning process :

1) raw transcription frequency maps
2) unvalidated clean transcriptions frequency maps
3) clean transcription frequency maps


In [ ]:
import platform

from lexical_benchmark import settings
from lexical_benchmark.datasets import stella
from lexical_benchmark.utils import DatasetCleaner, DictionairyCleaner, timed_status

assert (
    platform.node() in settings.PATH.KNOWN_HOSTS
), "Not in a known device, must provide custom PATH locations"

dataset = stella.STELATranscriptDataset()

with timed_status(status="Computing word frequencies of units...", complete_status="Succesfully computed all word frequencies !"):
    for item in dataset.iter_all():
        _ = item.clean_meta.raw_word_frequencies()
        _ = item.clean_meta.clean_word_frequencies()
        _ = item.clean_meta.rejected_word_frequencies()

## Computing cleanup stats

#### Global Rejection rates

We aggregate word frequency of the raw/clean/rejected accross the whole dataset & compute word rejection rate

In [1]:
import platform

import pandas as pd
from IPython.display import display
from lexical_benchmark import settings
from lexical_benchmark.datasets import stella
from lexical_benchmark.utils import timed_status

assert (
    platform.node() in settings.PATH.KNOWN_HOSTS
), "Not in a known device, must provide custom PATH locations"


freq_builder = stella.STELAWordFrequencies()

with timed_status(status="Computing word frequencies of units...", complete_status="Succesfully computed all word frequencies !"):
    raw_wf = freq_builder.word_frequencies(word_type="raw")["EN"]
    rejected_wf = freq_builder.word_frequencies(word_type="rejected")["EN"]
    clean_wf = freq_builder.word_frequencies(word_type="clean")["EN"]
    df = pd.DataFrame([
        {"set": "raw", "tokens": raw_wf["freq"].sum(), "types": len(raw_wf["word"])},
        {"set": "rejected", "tokens": rejected_wf["freq"].sum(), "types": len(rejected_wf["word"])},
        {"set": "clean", "tokens": clean_wf["freq"].sum(), "types": len(clean_wf["word"])},
    ])
    type_rejection_rate = len(rejected_wf["word"]) / len(raw_wf["word"])
    token_rejection_rate = rejected_wf["freq"].sum() / raw_wf["freq"].sum()


display(
    df.style.format({
        "tokens": "{:_}",
        "types": "{:_}",
    })
)
print(f"""
Token rejection rate across the STELATranscription/EN dataset is {token_rejection_rate:.2%}
Type  rejection rate across the STELATranscription/EN dataset is {type_rejection_rate:.2%}
""")

Output()

Succesfully computed all word frequencies ! (Total time: 2 minutes and 39 seconds)

,set,tokens,types
0,raw,294_109_819,336_025
1,rejected,4_445_425,203_824
2,clean,289_663_438,132_199



Token rejection rate across the STELATranscription/EN dataset is 1.51%
Type  rejection rate across the STELATranscription/EN dataset is 60.66%



#### Section Rejection rates

To be able to explain the high variaty of content into the 1.51% of the dataset.

We will display word frequency of the raw/clean/rejected and compute rejection rate for each section of the dataset.

In [10]:
import platform

import pandas as pd
from lexical_benchmark import settings
from lexical_benchmark.datasets import stella
from lexical_benchmark.utils import ipython_utils, timed_status

assert (
    platform.node() in settings.PATH.KNOWN_HOSTS
), "Not in a known device, must provide custom PATH locations"


freq_builder = stella.STELAWordFrequencies()

with timed_status(status="Computing word frequencies of units...", complete_status="Succesfully computed all word frequencies !"):
    token_stats = []
    type_stats = []
    for lang in freq_builder.nav.languages:
        for hour in freq_builder.nav.hour_splits:
            clean_wf = freq_builder.word_frequencies_by_split(lang, hour, "clean")
            rejected_wf = freq_builder.word_frequencies_by_split(lang, hour, "rejected")
            raw_wf = freq_builder.word_frequencies_by_split(lang, hour, "raw")
            for section in freq_builder.nav.sections(lang, hour):
                section_id = f"{lang}_{hour}_{section}"
                type_stats.append(
                    {
                        "section": f"{lang}/{hour}/{section}",
                        "raw": len(raw_wf[section_id]["word"]),
                        "rejected": len(rejected_wf[section_id]["word"]),
                        "clean": len(clean_wf[section_id]["word"]),
                        "rejection_rate": len(rejected_wf[section_id]["word"]) / len(raw_wf[section_id]["word"]),
                        "acceptance_rate": len(clean_wf[section_id]["word"]) / len(raw_wf[section_id]["word"]),
                    }
                )
                raw_tokens_sum = raw_wf[section_id]["freq"].sum()
                clean_tokens_sum = clean_wf[section_id]["freq"].sum()
                rejected_tokens_sum = rejected_wf[section_id]["freq"].sum()
                token_stats.append(
                    {
                        "section": f"{lang}/{hour}/{section}",
                        "raw": raw_tokens_sum,
                        "rejected": rejected_tokens_sum,
                        "clean": clean_tokens_sum,
                        "rejection_rate": rejected_tokens_sum / raw_tokens_sum,
                        "acceptance_rate": clean_tokens_sum / raw_tokens_sum,
                    }
                )

    token_stats_df = pd.DataFrame(token_stats)
    type_stats_df = pd.DataFrame(type_stats)

formatting = {
    "raw": "{:_}",
    "rejected": "{:_}",
    "clean": "{:_}",
    "rejection_rate": "{:.4%}",
    "acceptance_rate": "{:.4%}",
}
ipython_utils.display_side_by_side([token_stats_df.style.format(formatting), type_stats_df.style.format(formatting)], captions=["Token Stats", "Type Stats"])

Output()

Succesfully computed all word frequencies ! (Total time: 9 seconds)

,section,raw,rejected,clean,rejection_rate,acceptance_rate
0,EN/50h/00,482_361,3_003,479_358,0.6226%,99.3774%
1,EN/50h/01,693_662,5_712,687_950,0.8235%,99.1765%
2,EN/50h/02,555_435,7_265,548_170,1.3080%,98.6920%
3,EN/50h/03,561_927,4_286,557_641,0.7627%,99.2373%
4,EN/50h/04,579_713,4_461,575_252,0.7695%,99.2305%
5,EN/50h/05,466_807,5_023,461_784,1.0760%,98.9240%
6,EN/50h/06,450_330,4_556,445_774,1.0117%,98.9883%
7,EN/50h/07,770_586,2_920,767_666,0.3789%,99.6211%
8,EN/50h/08,542_539,6_411,536_128,1.1817%,98.8183%
9,EN/50h/09,429_294,2_301,426_993,0.5360%,99.4640%


### Computing Block Averaging

To calculate word rejection rate in the dataset, we use the method of 
cutting each split into chunk of a specific size (16k tokens per chunk), and then proceed to calculate 
the rejection rate. We do this to allow verification of the averages and to allow comparisons with CHILDES 
as the CHILDES & others datasets do not have the same size. 


### 160k block size

In [2]:
import collections
import platform

import pandas as pd
from IPython.display import display_html
from lexical_benchmark import settings
from lexical_benchmark.datasets import stella
from lexical_benchmark.datasets import utils as dataset_utils
from lexical_benchmark.stats import block_average
from lexical_benchmark.utils import ipython_utils, timed_status

assert (
    platform.node() in settings.PATH.KNOWN_HOSTS
), "Not in a known device, must provide custom PATH locations"


CHUNK_SIZE = 160_000
dataset = stella.STELATranscriptDataset()
rj_by_section = {}
tables_160k = collections.defaultdict(list)
tables_with_sums_160k = collections.defaultdict(list)
dictionairies = {"EN": dataset_utils.Lexicon(lang="EN")}

with timed_status(
    status=f"Calculating Rejection Rates of StellaRaw({CHUNK_SIZE})...",
    complete_status=f"Completed ({CHUNK_SIZE})!",
):
    for item in dataset.iter_all():
        word_list = item.processed_raw_transcription.read_tokenized()
        word_chunk_list = block_average.split_and_fill_chunks(word_list, chunk_size=CHUNK_SIZE)
        rj_rate = block_average.calculate_block_word_filtering_rates(
            word_chunk_list, lexicon=dictionairies.get(item.lang)
        )
        rj_by_section[item.section_id] = rj_rate
        tables_160k[f"{item.lang}/{item.hour_split}"].append(
            {"Section": item.section, **rj_rate.view("table_median")}
        )
        tables_with_sums_160k[f"{item.lang}/{item.hour_split}"].append(
            {"Section": item.section, **rj_rate.view("table_median_with_sums")}
        )

    # Build dataframes
    tables_160k = {f"{key}": pd.DataFrame(rows) for key, rows in dict(tables_160k).items()}
    tables_with_sums_160k = {f"{key}": pd.DataFrame(rows) for key, rows in dict(tables_with_sums_160k).items()}

Output()

Completed (160000)! (Total time: 4 minutes and 47 seconds)

### 16k block size

In [1]:
import collections
import platform

import pandas as pd
from IPython.display import display_html
from lexical_benchmark import settings
from lexical_benchmark.datasets import stella
from lexical_benchmark.datasets import utils as dataset_utils
from lexical_benchmark.stats import block_average
from lexical_benchmark.utils import ipython_utils, timed_status

assert (
    platform.node() in settings.PATH.KNOWN_HOSTS
), "Not in a known device, must provide custom PATH locations"


CHUNK_SIZE = 16_000
dataset = stella.STELATranscriptDataset()
rj_by_section = {}
tables_16k = collections.defaultdict(list)
tables_with_sums_16k = collections.defaultdict(list)
dictionairies = {
    "EN": dataset_utils.Lexicon(lang="EN")
}

with timed_status(
    status=f"Calculating Rejection Rates of StellaRaw({CHUNK_SIZE})...",
    complete_status=f"Completed ({CHUNK_SIZE})!"
):
    for item in dataset.iter_all():
        word_list = item.processed_raw_transcription.read_tokenized()
        word_chunk_list = block_average.split_and_fill_chunks(
            word_list, chunk_size=CHUNK_SIZE
        )
        rj_rate = block_average.calculate_block_word_filtering_rates(
            word_chunk_list, lexicon=dictionairies.get(item.lang)
        )
        rj_by_section[item.section_id] = rj_rate
        tables_16k[f"{item.lang}/{item.hour_split}"].append({"Section": item.section, **rj_rate.view("table_median")})
        tables_with_sums_16k[f"{item.lang}/{item.hour_split}"].append(
            {"Section": item.section, **rj_rate.view("table_median_with_sums")}
        )

    # Build dataframes
    tables_16k = {f"{key}": pd.DataFrame(rows) for key, rows in dict(tables_16k).items()}
    tables_with_sums_16k = {f"{key}": pd.DataFrame(rows) for key, rows in dict(tables_with_sums_16k).items()}

Output()

Completed (16000)! (Total time: 4 minutes and 45 seconds)

In [3]:
# Display results
display_html("<h3> 160k block averages.</h3>")
ipython_utils.display_dataframes(
    tables_with_sums_160k,
    custom_format={
        'Tokens Raw': '{:,}',
        'Tokens Rejected': '{:,}',
        'Token Rejection': '{:.4%}',
        'Tokens Accepted': '{:,}',
        'Token Acceptance': '{:.4%}',
        'Types': '{:,}',
        'Types Rejected': '{:,}',
        'Type Rejection': '{:.4%}',
        'Types Accepted': '{:,}',
        'Type Acceptance': '{:.4%}'
    },
    delimiter=True,
    delimit_color="black",
    delimiter_size="5px",
    delimited_columns=[1, 2, 4, 6, 7, 9, 11]
)

### EN/50h

,Section,Tokens Raw,Tokens Rejected,Token Rejection,Tokens Accepted,Token Acceptance,Types,Types Rejected,Type Rejection,Types Accepted,Type Acceptance
0,00,"480,000","2,991",0.6231%,"477,009",99.3769%,"33,053","1,460",4.2796%,"31,593",95.7204%
1,01,"640,000","5,534",0.8647%,"634,466",99.1353%,"40,070","1,689",4.1992%,"38,381",95.8008%
2,02,"480,000","6,221",1.2960%,"473,779",98.7040%,"37,012","4,534",11.8896%,"32,478",88.1104%
3,03,"480,000","4,081",0.8502%,"475,919",99.1498%,"36,047","2,014",5.5363%,"34,033",94.4637%
4,04,"480,000","4,251",0.8856%,"475,749",99.1144%,"34,666","2,682",7.8402%,"31,984",92.1598%
5,05,"320,000","2,871",0.8972%,"317,129",99.1028%,"24,231","1,016",4.1698%,"23,215",95.8302%
6,06,"320,000","3,267",1.0209%,"316,733",98.9791%,"22,453",314,1.4109%,"22,139",98.5891%
7,07,"640,000","2,692",0.4206%,"637,308",99.5794%,"35,969",649,1.6757%,"35,320",98.3243%
8,08,"480,000","6,064",1.2633%,"473,936",98.7367%,"37,449","1,352",3.2908%,"36,097",96.7092%
9,09,"320,000","2,087",0.6522%,"317,913",99.3478%,"23,487",764,3.2474%,"22,723",96.7526%


### EN/100h

,Section,Tokens Raw,Tokens Rejected,Token Rejection,Tokens Accepted,Token Acceptance,Types,Types Rejected,Type Rejection,Types Accepted,Type Acceptance
0,00,"1,120,000","8,347",0.7453%,"1,111,653",99.2547%,"72,987","3,028",3.9542%,"69,959",96.0458%
1,01,"960,000","10,118",1.0540%,"949,882",98.9460%,"72,496","6,386",8.4030%,"66,110",91.5970%
2,02,"960,000","8,894",0.9265%,"951,106",99.0735%,"68,597","3,919",5.6953%,"64,678",94.3047%
3,03,"1,120,000","6,635",0.5924%,"1,113,365",99.4076%,"71,152","1,324",1.7874%,"69,828",98.2126%
4,04,"960,000","8,558",0.8915%,"951,442",99.1085%,"72,648","2,130",2.7929%,"70,518",97.2071%
5,05,"2,080,000","19,432",0.9342%,"2,060,568",99.0658%,"121,447","8,658",7.4406%,"112,789",92.5594%
6,06,"1,280,000","8,320",0.6500%,"1,271,680",99.3500%,"95,357","2,800",2.8236%,"92,557",97.1764%
7,07,"1,120,000","7,779",0.6946%,"1,112,221",99.3054%,"85,014","2,065",2.3622%,"82,949",97.6378%
8,08,"1,120,000","28,123",2.5110%,"1,091,877",97.4890%,"98,744","18,751",13.7663%,"79,993",86.2337%
9,09,"960,000","6,257",0.6518%,"953,743",99.3482%,"69,628","2,628",3.7198%,"67,000",96.2802%


### EN/200h

...

,Section,Tokens Raw,Tokens Rejected,Token Rejection,Tokens Accepted,Token Acceptance,Types,Types Rejected,Type Rejection,Types Accepted,Type Acceptance
0,00,"10,080,000","80,965",0.8032%,"9,999,035",99.1968%,"700,906","32,073",4.5609%,"668,833",95.4391%
1,01,"11,360,000","202,930",1.7864%,"11,157,070",98.2136%,"908,199","121,540",10.4378%,"786,659",89.5622%
2,02,"11,520,000","260,069",2.2575%,"11,259,931",97.7425%,"832,241","82,119",9.2089%,"750,122",90.7911%
3,03,"8,800,000","88,624",1.0071%,"8,711,376",98.9929%,"689,897","48,803",6.5491%,"641,094",93.4509%


### EN/1600h

,Section,Tokens Raw,Tokens Rejected,Token Rejection,Tokens Accepted,Token Acceptance,Types,Types Rejected,Type Rejection,Types Accepted,Type Acceptance
0,00,"21,440,000","284,333",1.3262%,"21,155,667",98.6738%,"1,613,757","155,167",7.9328%,"1,458,590",92.0672%
1,01,"20,320,000","347,824",1.7117%,"19,972,176",98.2883%,"1,522,383","130,305",8.1179%,"1,392,078",91.8821%


### EN/3200h

,Section,Tokens Raw,Tokens Rejected,Token Rejection,Tokens Accepted,Token Acceptance,Types,Types Rejected,Type Rejection,Types Accepted,Type Acceptance
0,00,"41,920,000","633,976",1.5123%,"41,286,024",98.4877%,"3,150,462","286,086",7.9095%,"2,864,376",92.0905%


In [2]:
# Display results
display_html("<h3> 16k block averages.</h3>")
ipython_utils.display_dataframes(
    tables_with_sums_16k,
    custom_format={
        'Tokens Raw': '{:,}',
        'Tokens Rejected': '{:,}',
        'Token Rejection': '{:.4%}',
        'Tokens Accepted': '{:,}',
        'Token Acceptance': '{:.4%}',
        'Types': '{:,}',
        'Types Rejected': '{:,}',
        'Type Rejection': '{:.4%}',
        'Types Accepted': '{:,}',
        'Type Acceptance': '{:.4%}'
    },
    delimiter=True,
    delimit_color="black",
    delimiter_size="5px",
    delimited_columns=[1, 2, 4, 6, 7, 9, 11]
)

### EN/50h


,Section,Tokens Raw,Tokens Rejected,Token Rejection,Tokens Accepted,Token Acceptance,Types,Types Rejected,Type Rejection,Types Accepted,Type Acceptance
0,00,"480,000","2,991",0.6231%,"477,009",99.3769%,"82,780","1,647",1.9397%,"81,133",98.0603%
1,01,"688,000","5,688",0.8267%,"682,312",99.1733%,"105,453","2,051",1.9750%,"103,402",98.0250%
2,02,"544,000","7,034",1.2930%,"536,966",98.7070%,"103,332","5,923",5.8356%,"97,409",94.1644%
3,03,"560,000","4,254",0.7596%,"555,746",99.2404%,"102,952","2,472",2.4295%,"100,480",97.5705%
4,04,"576,000","4,460",0.7743%,"571,540",99.2257%,"96,439","3,190",3.6050%,"93,249",96.3950%
5,05,"464,000","5,021",1.0821%,"458,979",98.9179%,"85,910","1,681",1.9928%,"84,229",98.0072%
6,06,"448,000","4,472",0.9982%,"443,528",99.0018%,"84,742","1,006",1.1593%,"83,736",98.8407%
7,07,"768,000","2,899",0.3775%,"765,101",99.6225%,"114,187",919,0.7520%,"113,268",99.2480%
8,08,"528,000","6,349",1.2025%,"521,651",98.7975%,"98,880","1,687",1.6887%,"97,193",98.3113%
9,09,"416,000","2,239",0.5382%,"413,761",99.4618%,"75,718",962,1.2270%,"74,756",98.7730%



### EN/100h


,Section,Tokens Raw,Tokens Rejected,Token Rejection,Tokens Accepted,Token Acceptance,Types,Types Rejected,Type Rejection,Types Accepted,Type Acceptance
0,00,"1,168,000","8,639",0.7396%,"1,159,361",99.2604%,"188,195","3,675",1.9334%,"184,520",98.0666%
1,01,"1,104,000","11,287",1.0224%,"1,092,713",98.9776%,"206,898","8,408",4.0778%,"198,490",95.9222%
2,02,"1,040,000","9,476",0.9112%,"1,030,524",99.0888%,"183,254","4,861",2.8500%,"178,393",97.1500%
3,03,"1,216,000","7,310",0.6012%,"1,208,690",99.3988%,"199,291","1,908",0.8937%,"197,383",99.1063%
4,04,"960,000","8,558",0.8915%,"951,442",99.1085%,"177,799","2,565",1.4205%,"175,234",98.5795%
5,05,"2,144,000","19,515",0.9102%,"2,124,485",99.0898%,"321,341","10,446",3.2154%,"310,895",96.7846%
6,06,"1,392,000","8,967",0.6442%,"1,383,033",99.3558%,"265,863","3,890",1.3822%,"261,973",98.6178%
7,07,"1,120,000","7,779",0.6946%,"1,112,221",99.3054%,"215,220","2,481",1.1094%,"212,739",98.8906%
8,08,"1,264,000","29,886",2.3644%,"1,234,114",97.6356%,"246,039","21,614",8.1174%,"224,425",91.8826%
9,09,"1,072,000","6,719",0.6268%,"1,065,281",99.3732%,"192,840","3,312",1.6962%,"189,528",98.3038%



### EN/200h


...

,Section,Tokens Raw,Tokens Rejected,Token Rejection,Tokens Accepted,Token Acceptance,Types,Types Rejected,Type Rejection,Types Accepted,Type Acceptance
0,00,"10,208,000","82,463",0.8078%,"10,125,537",99.1922%,"1,770,206","38,829",2.2101%,"1,731,377",97.7899%
1,01,"11,360,000","202,930",1.7864%,"11,157,070",98.2136%,"2,113,173","141,761",5.7606%,"1,971,412",94.2394%
2,02,"11,552,000","260,281",2.2531%,"11,291,719",97.7469%,"2,018,303","119,062",6.0497%,"1,899,241",93.9503%
3,03,"8,896,000","89,278",1.0036%,"8,806,722",98.9964%,"1,662,895","56,447",3.3641%,"1,606,448",96.6359%



### EN/1600h


,Section,Tokens Raw,Tokens Rejected,Token Rejection,Tokens Accepted,Token Acceptance,Types,Types Rejected,Type Rejection,Types Accepted,Type Acceptance
0,00,"21,568,000","285,500",1.3237%,"21,282,500",98.6763%,"3,882,180","180,667",4.0912%,"3,701,513",95.9088%
1,01,"20,448,000","349,524",1.7093%,"20,098,476",98.2907%,"3,683,107","175,370",4.8803%,"3,507,737",95.1197%



### EN/3200h


,Section,Tokens Raw,Tokens Rejected,Token Rejection,Tokens Accepted,Token Acceptance,Types,Types Rejected,Type Rejection,Types Accepted,Type Acceptance
0,00,"42,016,000","635,027",1.5114%,"41,380,973",98.4886%,"7,566,511","355,900",4.4665%,"7,210,611",95.5335%
